# Alpine Crest Private-Banking Feature Engineering

Build reusable private-banking feature families from tiny Alpine Crest Private Bank data using the package feature library and the SQLite-first learner path. The generated data is synthetic and the feature work below uses learner-facing views.


In [ ]:
from pathlib import Path

import pandas as pd

from banking_fraud_lab import (
    build_learner_facing_views,
    build_private_banking_features,
    generate_private_banking_transaction_fraud_world,
    load_tables_to_sqlite,
)
from banking_fraud_lab.features import PRIVATE_BANKING_FEATURE_FAMILIES
from banking_fraud_lab.schema import PATTERN_IDS, PROTECTED_SCENARIO_ANSWER_KEYS

pd.set_option("display.max_columns", 40)


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find the repository root.")


repo_root = find_repo_root(Path.cwd())


## Generate Learner-Facing Data

The canonical generator retains protected scenario answer keys for internal validation. The learner-facing tables loaded below exclude that protected table.


In [ ]:
tables = generate_private_banking_transaction_fraud_world(
    seed=42,
    scenario_prevalence=0.2,
)
learner_tables = build_learner_facing_views(tables)

assert PROTECTED_SCENARIO_ANSWER_KEYS in tables
assert PROTECTED_SCENARIO_ANSWER_KEYS not in learner_tables

learner_table_summary = (
    pd.DataFrame(
        {
            "table": list(learner_tables),
            "rows": [len(frame) for frame in learner_tables.values()],
        }
    )
    .sort_values("table", kind="stable")
    .reset_index(drop=True)
)
learner_table_summary


## Map Feature Families To Detection Patterns

The feature-family registry links every output column to a Detection pattern identifier from the canonical vocabulary.


In [ ]:
feature_family_map = pd.DataFrame(
    [
        {
            "family_id": spec.family_id,
            "display_name": spec.display_name,
            "detection_pattern_id": spec.detection_pattern_id,
            "output_columns": ", ".join(spec.output_columns),
        }
        for spec in PRIVATE_BANKING_FEATURE_FAMILIES
    ]
)

assert set(feature_family_map["detection_pattern_id"]).issubset(set(PATTERN_IDS))
assert set(feature_family_map["detection_pattern_id"]) <= {
    "pb_high_value_movement",
    "pb_transaction_fraud",
}
feature_family_map


## Build Python Feature Frame

`build_private_banking_features()` calculates the complete Alpine Crest transaction-level feature frame from canonical tables.


In [ ]:
feature_frame = build_private_banking_features(learner_tables)
expected_feature_columns = {
    output_column
    for spec in PRIVATE_BANKING_FEATURE_FAMILIES
    for output_column in spec.output_columns
}

assert expected_feature_columns <= set(feature_frame.columns)
assert PROTECTED_SCENARIO_ANSWER_KEYS not in feature_frame.columns
assert set(feature_frame["institution_name"]) == {"Alpine Crest Private Bank"}

feature_columns_to_review = [
    "transaction_id",
    "banking_relationship_id",
    "relationship_manager_code",
    "amount_chf",
    "amount_to_aum_ratio",
    "amount_to_relationship_baseline_ratio",
    "is_new_counterparty",
    "is_off_hours",
    "is_cross_border",
    "relationship_txn_count_7d",
    "relationship_txn_count_30d",
    "rm_alert_share",
]
feature_frame.loc[:, feature_columns_to_review].head(8)


## Run The SQLite Feature Exercises

The same learner-facing tables can be loaded into SQLite and queried through the representative private-banking SQL exercises.


In [ ]:
connection = load_tables_to_sqlite(learner_tables, ":memory:")

sqlite_tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name",
    connection,
)
assert PROTECTED_SCENARIO_ANSWER_KEYS not in set(sqlite_tables["name"])

sql_paths = {
    "value_features": repo_root / "sql/examples/06_private_banking_value_features.sql",
    "context_features": repo_root / "sql/examples/07_private_banking_context_features.sql",
    "relationship_features": repo_root / "sql/examples/08_private_banking_relationship_features.sql",
}
sql_results = {
    name: pd.read_sql_query(path.read_text(encoding="utf-8"), connection)
    for name, path in sql_paths.items()
}

for result in sql_results.values():
    assert len(result) == len(feature_frame)

sql_result_summary = pd.DataFrame(
    [
        {
            "exercise": name,
            "rows": len(result),
            "columns": ", ".join(result.columns[:6]),
        }
        for name, result in sql_results.items()
    ]
)
sql_result_summary


## Compare Python And SQL Outputs

The SQL examples are learner-readable versions of the same feature ideas, so this smoke check compares representative columns across both paths.


In [ ]:
python_value_features = feature_frame.assign(
    amount_to_aum_ratio=lambda frame: frame["amount_to_aum_ratio"].round(6),
    amount_to_relationship_baseline_ratio=lambda frame: frame[
        "amount_to_relationship_baseline_ratio"
    ].round(4),
)
value_comparison = python_value_features.merge(
    sql_results["value_features"][
        [
            "transaction_id",
            "amount_to_aum_ratio",
            "amount_to_relationship_baseline_ratio",
        ]
    ],
    on="transaction_id",
    suffixes=("_python", "_sql"),
    validate="one_to_one",
)

for column in (
    "amount_to_aum_ratio",
    "amount_to_relationship_baseline_ratio",
):
    assert (
        value_comparison[f"{column}_python"]
        .sub(value_comparison[f"{column}_sql"])
        .abs()
        .le(0.0001)
        .all()
    )

context_columns = ["is_off_hours", "is_cross_border"]
context_comparison = feature_frame.merge(
    sql_results["context_features"][["transaction_id", *context_columns]],
    on="transaction_id",
    suffixes=("_python", "_sql"),
    validate="one_to_one",
)
for column in context_columns:
    assert (context_comparison[f"{column}_python"] == context_comparison[f"{column}_sql"]).all()

relationship_columns = ["is_new_counterparty", "rm_alert_share"]
relationship_comparison = feature_frame.merge(
    sql_results["relationship_features"][["transaction_id", *relationship_columns]],
    on="transaction_id",
    suffixes=("_python", "_sql"),
    validate="one_to_one",
)
assert (
    relationship_comparison["is_new_counterparty_python"]
    == relationship_comparison["is_new_counterparty_sql"]
).all()
assert (
    relationship_comparison["rm_alert_share_python"]
    .round(4)
    .sub(relationship_comparison["rm_alert_share_sql"])
    .abs()
    .le(0.0001)
    .all()
)

feature_frame.loc[:, feature_columns_to_review].describe(include="all")


In [ ]:
connection.close()
